# Stage 1 — Ingest & auto-label new images
Drop new unlabelled images in `/dataset/incoming/`. This proposes detections
+species with the current ONNX pipeline, splits by confidence, and writes
Label Studio import tasks for human verification.

In [1]:
import sys; sys.path.insert(0,'/workspace/lib')
import os, json, glob
from inference_lib import Pipeline
from labelstudio_lib import to_ls_task, build_label_config
from PIL import Image

In [4]:
# --- config: point at the current best models ---
DETECTOR   = '/workspace/cai_uk_mammals_yolo10x_v1.0.0.onnx'   # <-- EDIT: your YOLO detector ONNX
CLASSIFIER = sorted(glob.glob('/dataset/models/**/*.onnx', recursive=True))[-1]  # latest
INCOMING   = '/dataset/incoming'
REVIEW_CONF = 0.50   # below this -> flagged for careful review
print('Detector  :', DETECTOR)
print('Classifier:', CLASSIFIER)

Detector  : /workspace/cai_uk_mammals_yolo10x_v1.0.0.onnx
Classifier: /dataset/models/v1.1/deepfaune_uk_v1.1.onnx


In [6]:
pipe = Pipeline(DETECTOR, CLASSIFIER)
imgs = sorted(f for f in os.listdir(INCOMING) if f.lower().endswith(('.jpg','.jpeg','.png')))
print(len(imgs), 'incoming images')

2026-07-02 10:13:10.207575784 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 24 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


62 incoming images


In [7]:
# Run inference, build LS tasks. LS serves images via local-files using the
# path RELATIVE to /label-studio/files (which is mounted to ./dataset).
tasks, hi, lo = [], 0, 0
for fn in imgs:
    fp = os.path.join(INCOMING, fn)
    frame, dets = pipe.infer(fp)
    ls_path = f'/data/local-files/?d=incoming/{fn}'
    tasks.append(to_ls_task(ls_path, frame.size, dets))
    # confidence bookkeeping for triage
    for d in dets:
        if d['kind']=='animal':
            if d['species_conf'] is not None and d['species_conf'] < REVIEW_CONF: lo+=1
            else: hi+=1
print(f'tasks: {len(tasks)} | high-conf animal dets: {hi} | low-conf (review): {lo}')

tasks: 62 | high-conf animal dets: 30 | low-conf (review): 1


In [9]:
import numpy as np

def _to_native(o):
    if isinstance(o, (np.floating,)):  return float(o)
    if isinstance(o, (np.integer,)):   return int(o)
    if isinstance(o, np.ndarray):      return o.tolist()
    raise TypeError(f'{type(o)} not serializable')

os.makedirs('/dataset/ls_import', exist_ok=True)
json.dump(tasks, open('/dataset/ls_import/tasks.json','w'), indent=1, default=_to_native)
open('/dataset/ls_import/label_config.xml','w').write(build_label_config())
print('Wrote /dataset/ls_import/tasks.json and label_config.xml')

Wrote /dataset/ls_import/tasks.json and label_config.xml


## Next: verify in Label Studio (Stage 2)
1. Open http://localhost:8080 and create/login.
2. Create a project; paste `ls_import/label_config.xml` as the labeling config.
3. In project settings enable **Local Storage**, set root to the mounted
   `/label-studio/files`, and add `incoming/` so images are served.
4. Import `ls_import/tasks.json` (brings boxes+species as pre-annotations).
5. Review: approve correct ones, fix boxes/labels on the rest. Focus on the
   low-confidence detections flagged above.
6. Export the project as **JSON** to `/dataset/ls_export/export.json`.
Then run Stage 3.